# Faster R-CNN Small Object Detection — xView (Paper Replication)
### Yuan et al., *An empirical analysis of deep learning methods for small object detection from satellite imagery*
### Faster R-CNN portion · 6-Fold Cross-Validation · Local GPU build — interruption-safe, 8 GB GPU / 15 GB RAM friendly

---

This notebook replicates the **Faster R-CNN portion** of:

> Yuan, X., Chakravarty, A., Lichtenberg, E. M., Gu, L., Wei, Z., & Chen, T.
> *An empirical analysis of deep learning methods for small object detection from satellite imagery.*
> *Expert Systems With Applications* 307 (2026) 131061.

The paper compares six deep-learning detectors (YOLOv11, Faster R-CNN, SSD,
Cascade R-CNN, Deformable DETR, RT-DETR) on three high-resolution satellite
datasets. This notebook focuses on **Faster R-CNN with a ResNet-50 + FPN
backbone**, which the paper reports as the strongest two-stage detector with
**consistent detection coverage** (Table 13).

---

## Interruption-safe pipeline

Everything in the notebook is built to survive kernel restarts, machine
reboots, and accidental cell re-runs. The key idea: **state lives on disk,
not in memory**.

| What\'s saved | Where | When |
|---|---|---|
| Model + optimizer + AMP scaler + epoch + patience counter + history | `outputs_frcnn/checkpoints/frcnn_fold{N}/last.pt` | **Every `CHECKPOINT_EVERY_N_STEPS` optimizer steps** AND at the end of every epoch |
| Best-AP50:95 weights (final-eval source) | `outputs_frcnn/checkpoints/frcnn_fold{N}/best.pt` | Whenever AP50:95 improves by > `MIN_DELTA` |
| Per-epoch history (loss, AP, P/R/F1) | `outputs_frcnn/checkpoints/frcnn_fold{N}/history.csv` | Every epoch |
| Per-fold final metrics | `outputs_frcnn/metrics/fold{N}_results.csv` | When a fold finishes |
| Fold-level status (pending/running/completed/failed/interrupted) | `outputs_frcnn/metrics/training_state.json` | After every checkpoint, atomic write |
| Tile cache | `outputs/tiles/` (shared with YOLO nb) | Iteratively flushed every 5 source images |
| Fold splits | `outputs/metrics/fold_splits.json` (shared with YOLO nb) | Once, deterministic, cached |

**If your kernel/PC dies, just re-run the notebook.** The training loop will:
1. Load `last.pt` (which captures the *latest* in-progress state, not just the
   last completed epoch),
2. Restart the current epoch with the already-advanced weights/optimizer
   state — the model effectively keeps the gradient updates already applied,
3. Skip any folds whose `best.pt` + per-fold CSV already exist,
4. Continue early-stopping with the same patience counter you had before the
   crash.

All checkpoint writes are **atomic** (`.tmp` → `os.replace`), so a crash
mid-write can never corrupt the checkpoint file.

---

## Paper data referenced in this notebook

### Table 5 — Tuned hyperparameters (Faster R-CNN row)
| Param | Value | Where used |
|---|---|---|
| Backbone | ResNet-50 + FPN | `build_faster_rcnn` |
| Optimizer | SGD | `train_fold` |
| Learning rate | 38.10 × 10⁻⁴ = **3.81e-3** | `LEARNING_RATE` |
| Momentum | **0.90** | `MOMENTUM` |
| Weight decay | 0.0285 × 10⁻⁴ = **2.85e-6** | `WEIGHT_DECAY` |
| Batch size | **16** (we use 2 × grad-accum 8) | `BATCH_SIZE × GRAD_ACCUM` |
| Range of anchor box size (RABS) | **10 → 90 px** | `ANCHOR_MIN_PX`, `ANCHOR_MAX_PX` |
| Aspect ratios (short/long ∈ (0,1]) | **0.75, 0.85, 0.95** | `ANCHOR_ASPECTS` (we also include the reciprocals so rotated vehicles can match either orientation) |

### Table 8 — RPN region-proposal threshold (Section 4.3)
Paper finds **0.5** is optimal for small-object regimes (vs the torchvision
default 0.7). Hard-coded as `rpn_fg_iou_thresh=0.5` in `build_faster_rcnn`.

### Table 12 — Frozen backbone layers
Paper: **49 frozen layers** for Faster R-CNN. With torchvision\'s `trainable_layers ∈ [0,5]`, we freeze the first two stages (`trainable_layers=3`), which corresponds to roughly the first 49 conv layers of ResNet-50.

### Section 4.1 — Training schedule
Paper: **10,000 iterations** per fold. At our effective batch 16 with
~7.5k training tiles per fold, one epoch ≈ 469 iterations, so 10k iter ≈ 21
epochs. We cap at `EPOCHS=100` with `PATIENCE=15` and `MIN_DELTA=0.05` so the
network can train as long as it\'s still learning, then stop cleanly when it
plateaus.

### Section 3.1 — Small-object definition
Bounding-box area **10 ≤ area ≤ 1000 px²** for an object to count as
"small". Applied in the class-selection cell (`MIN_OBJ_PX`, `MAX_OBJ_PX`).

### Table 3 — Target xView classes
Four vehicle classes survive the small-object filter:
| xView class id | Name | FRCNN class id |
|---|---|---|
| 18 | Small Car | 1 |
| 17 | Passenger Vehicle | 2 |
| 20 | Pickup Truck | 3 |
| 21 | Utility Truck | 4 |

(Class 0 is reserved for **background** by Faster R-CNN convention.
The YOLO notebook uses 0..3 for these same four classes.)

### Section 3.1 / Table 4 — xView scale
- Original xView: 60 categories, ~1 M annotations across 846 high-resolution images
- After applying the small-object filter (10–1000 px²) and restricting to the 4 vehicle classes: ~270k objects across ~720 images, then tiled to 512×512 → ~45k training tiles
- Image GSD: ~30 cm/pixel

### Table 13 — xView 6-fold-CV results (target to reproduce)
| Detector | AP50 (± std) | AP50:95 (± std) | F1 @ IoU=0.5 (± std) |
|---|---|---|---|
| **Faster R-CNN** | **61.65 (± 3.22)** | **23.32 (± 1.30)** | **69.44 (± 2.07)** |
| YOLOv11 (reference) | 64.30 (± 2.84) | 26.78 (± 1.07) | 71.82 (± 1.71) |

The summary cell at the bottom of the notebook prints our mean ± std for
direct side-by-side comparison.


---
## 1. Configuration

### Quick start
1. **Kaggle credentials.** Drop `kaggle.json` (from *Kaggle Account → Create New API
   Token*) into the project root, or put it at `~/.kaggle/kaggle.json`. The
   notebook auto-downloads xView (`hassanmojab/xview-dataset`) on first run;
   later runs reuse the cached copy.
2. **Smoke test first.** Set `DEBUG_MODE = True` and run all cells (1 epoch
   on fold 0 with batch 1, ~10 min) to verify the pipeline end-to-end.
3. **Full run.** Set `DEBUG_MODE = False`, `CURRENT_FOLD = "all"`, then run
   all cells. Training auto-resumes from the last saved checkpoint.

### If something breaks
- **Crash mid-training?** Just rerun every cell. Each fold picks up from
  the most recent intra-epoch checkpoint (see `CHECKPOINT_EVERY_N_STEPS`
  below). Already-finished folds are skipped automatically.
- **CUDA out of memory?** Lower `BATCH_SIZE` (try 1) and double `GRAD_ACCUM`
  so the effective batch stays at 16.
- **Loss going NaN?** Set `AMP = False` (a few mixed-precision corner cases
  hit small detection models). The gradient clipping at 10.0 should already
  handle most LR-related blow-ups.
- **Want to restart a fold from scratch?** Delete its folder under
  `outputs_frcnn/checkpoints/frcnn_fold{N}/`.


In [ ]:
# !pip install numpy pandas matplotlib seaborn opencv-python-headless tqdm scikit-learn pyyaml torch torchvision torchmetrics pycocotools kaggle


In [ ]:
# !pip install --upgrade --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu124


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  1.1  EDIT THESE VARIABLES
# ══════════════════════════════════════════════════════════════════
#
#  Every other cell reads from this one — DO NOT redefine these
#  variables anywhere else. Re-run this cell after editing.
# ══════════════════════════════════════════════════════════════════

# ---- Mode ------------------------------------------------------------
# DEBUG_MODE = True   → 1-epoch / batch-1 / fold-0 smoke test (~10 min on
#                       8 GB GPU). Use this on the first ever run to verify
#                       paths, Kaggle creds, CUDA, tile cache, and that the
#                       loss decreases. Then flip to False.
DEBUG_MODE = False

# ---- Dataset (auto-downloaded from Kaggle) ---------------------------
# The xView dataset is fetched from Kaggle on first run and cached under
# LOCAL_DATASET_DIR. Subsequent runs reuse the cached copy (no network).

# KAGGLE_DATASET    = "hassanmojab/xview-dataset"
LOCAL_DATASET_DIR = "./dataset/xview"

# ---- Cross-validation (Paper Section 4.1: 6-fold) --------------------
NUM_FOLDS    = 6
# "all"          → runs every fold whose `best.pt` + per-fold CSV are missing
# 0..5 (int)     → force one fold; useful when re-running a single failed fold
# Already-completed folds are skipped automatically in both modes.
CURRENT_FOLD = "all"

# ══════════════════════════════════════════════════════════════════
#  Paper hyperparameters (Yuan et al. 2026, Table 5 — Faster R-CNN row)
#  These come from the Optuna search the authors performed; do not change
#  them unless you intend to deviate from the paper.
# ══════════════════════════════════════════════════════════════════

TILE_SIZE      = 512           # 512×512 tiles (Section 4.1)
TRAIN_RATIO    = 0.70          # Per-fold train/val split inside the train half
IMG_SIZE       = 512
FLIP_PROB      = 0.5           # Horizontal-flip augmentation (Section 4.1)
SEED           = 42

# ---- Optimisation (Table 5) ------------------------------------------
LEARNING_RATE  = 38.10e-4      # 3.81e-3  (Table 5)
MOMENTUM       = 0.90          # (Table 5)
WEIGHT_DECAY   = 0.0285e-4     # 2.85e-6  (Table 5)

# ---- Effective batch size (Table 5: batch=16) ------------------------
#  Paper uses batch=16 directly on a larger GPU. On an 8 GB card we
#  micro-batch and accumulate gradients to reach the same effective batch.
BATCH_SIZE     = 2             # micro-batch (2 × 512×512 tiles fits in 8 GB)
GRAD_ACCUM     = 8             # 2 × 8 = 16 effective batch (paper)

# ---- Anchor generator (Table 5 + Section 4.2) ------------------------
ANCHOR_MIN_PX  = 10            # Smallest anchor side length (px)
ANCHOR_MAX_PX  = 90            # Largest  anchor side length (px)  — RABS=90
# Paper aspect ratios are short/long ∈ (0, 1] = {0.75, 0.85, 0.95}.
# Vehicles in satellite imagery are arbitrarily rotated, so we ALSO add
# the reciprocals (1.05, 1.18, 1.33) so the detector can match a vehicle
# regardless of whether its long axis is horizontal or vertical.
ANCHOR_ASPECTS = [0.75, 0.85, 0.95, 1.05, 1.18, 1.33]

# ---- RPN / inference thresholds (Section 4.1 + Section 4.3) ----------
RPN_SCORE_THRESH = 0.0         # torchvision default; RPN keeps top-K then NMS
BOX_SCORE_THRESH = 0.05        # final-stage conf floor for COCO-style eval
NMS_IOU_THRESH   = 0.5         # standard
CONF_THRESHOLD   = 0.25        # display / overlay threshold

# ---- Frozen backbone layers (Table 12) -------------------------------
# Paper froze 49 conv layers for Faster R-CNN.  torchvision exposes the
# number of TRAINABLE backbone STAGES in [0, 5]; setting trainable=3
# freezes the first two ResNet-50 stages, which contains ~49 conv layers.
FROZEN_BACKBONE_LAYERS = 3

# ---- Small-object definition (Paper Section 3.1) ---------------------
MAX_OBJ_PX = 1000              # px²; > 1000 is "medium" or "large"
MIN_OBJ_PX = 10                # px²; < 10 is essentially noise

# ══════════════════════════════════════════════════════════════════
#  Training behaviour (you CAN tune these)
# ══════════════════════════════════════════════════════════════════

# ---- Resume / control flags ------------------------------------------
RESUME_TRAINING  = True        # Auto-resume from last.pt if present
SKIP_TRAINING    = False       # True = only run final eval on existing weights
FORCE_RETILE     = False       # True = wipe tile cache and rebuild
FORCE_RESPLIT    = False       # True = rebuild fold_splits.json even if cached

# ---- Training budget -------------------------------------------------
# Paper used 10,000 iterations per fold (Section 4.1). At our effective
# batch 16 with ~7.5k train tiles per fold, that\'s ≈ 21 epochs.
#
# Because each crash-resume costs you at most one partial epoch (intra-epoch
# checkpoints below), there is no downside to setting EPOCHS high and
# letting EARLY STOPPING do the work. We use:
#
#    EPOCHS    = 100   → ceiling, almost never reached in practice
#    PATIENCE  = 15    → stop after 15 epochs with no AP50:95 improvement
#    MIN_DELTA = 0.05  → improvements smaller than this (in AP50:95 percentage
#                        points) don\'t reset the patience counter, so the
#                        loop stops cleanly when training has plateaued
#                        instead of chasing 0.01-point noise for 14 more
#                        epochs.
EPOCHS    = 100
PATIENCE  = 15
MIN_DELTA = 0.05
EVAL_EVERY_EPOCH = True

# ---- Intra-epoch checkpointing ---------------------------------------
# Save the full training state (model + optimizer + scaler + history +
# patience counter) every N OPTIMIZER steps inside an epoch. This means a
# crash never loses more than ~N×effective_batch tiles worth of training.
#
# With BATCH_SIZE=2, GRAD_ACCUM=8, and 7.5k train tiles, one epoch is
# ~470 optimizer steps. CHECKPOINT_EVERY_N_STEPS=100 → roughly 5 saves per
# epoch, each ≈ 2 minutes of work. Set to 0 to disable intra-epoch saves
# (you\'ll still get end-of-epoch saves).
CHECKPOINT_EVERY_N_STEPS = 100

# ---- Hardware / memory knobs (15 GB RAM, 8 GB GPU) -------------------
NUM_WORKERS  = 2               # DataLoader workers; lower if RAM tight
AMP          = True            # Mixed-precision; halves GPU memory
DEVICE       = "cuda"          # auto-fallback to "cpu" if no CUDA

import os as _os
# Helps PyTorch avoid memory fragmentation on small GPUs.
_os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF",
                       "expandable_segments:True,max_split_size_mb:128")
_os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "0")

# ══════════════════════════════════════════════════════════════════
#  DEBUG overrides — cheap end-to-end smoke test
# ══════════════════════════════════════════════════════════════════
if DEBUG_MODE:
    EPOCHS                   = 1
    BATCH_SIZE               = 1
    GRAD_ACCUM               = 2
    CHECKPOINT_EVERY_N_STEPS = 50   # save more often in debug
    if isinstance(CURRENT_FOLD, str):
        CURRENT_FOLD = 0
    print("=" * 70)
    print("  DEBUG MODE: 1 epoch, fold 0 only, batch=1")
    print("=" * 70)
else:
    print("=" * 70)
    print(f"  FULL TRAINING   Folds: {NUM_FOLDS}")
    print(f"  Fold(s) to run  : {CURRENT_FOLD}")
    print(f"  Epochs (max)    : {EPOCHS}    Patience: {PATIENCE}    MinΔ: {MIN_DELTA}")
    print(f"  Effective batch : {BATCH_SIZE * GRAD_ACCUM}  ({BATCH_SIZE} × accum {GRAD_ACCUM})")
    print(f"  Save every      : {CHECKPOINT_EVERY_N_STEPS} optim steps "
          "(plus end-of-epoch + best-so-far)")
    print("=" * 70)

print(f"  LR / momentum   : {LEARNING_RATE:g} / {MOMENTUM}   WD: {WEIGHT_DECAY:g}")
print(f"  Anchors         : sizes {ANCHOR_MIN_PX}-{ANCHOR_MAX_PX}px, ratios {ANCHOR_ASPECTS}")
print(f"  Resume          : {RESUME_TRAINING}   AMP: {AMP}   Workers: {NUM_WORKERS}")


---
## 2. Environment Setup

Installs any missing packages, configures torch / OpenCV thread counts so
DataLoader workers don't oversubscribe CPUs, and probes the GPU.


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  2.1  Install + import + GPU probe + output directories
# ══════════════════════════════════════════════════════════════════
import os, sys, platform, subprocess, shutil, zipfile, time, gc, json, random, warnings
from pathlib import Path
from collections import defaultdict
from datetime import datetime

print("[2.1] Setting up environment ...")

# Lightweight pip install only when a package is missing — keeps re-runs fast.
def _pip(import_name, package_name=None):
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name or import_name],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )

for imp, pkg in [
    ("numpy", None), ("pandas", None), ("matplotlib", None), ("seaborn", None),
    ("cv2", "opencv-python-headless"), ("tqdm", None), ("sklearn", "scikit-learn"),
    ("yaml", "pyyaml"), ("torch", None), ("torchvision", None),
    ("torchmetrics", None), ("pycocotools", None),
]:
    _pip(imp, pkg)

import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")           # safe inside notebooks without a display
import matplotlib.pyplot as plt, seaborn as sns, cv2, yaml
import torch, torchvision
from tqdm.auto import tqdm
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchmetrics.detection import MeanAveragePrecision

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "font.size": 11})
sns.set_style("whitegrid")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Cap thread counts so DataLoader workers don't fight the main thread.
try:
    torch.set_num_threads(max(1, (os.cpu_count() or 4) // 2))
except Exception:
    pass
torch.backends.cudnn.benchmark = True
try:
    cv2.setNumThreads(1)
except Exception:
    pass

# ---- GPU probe -------------------------------------------------------
HW = {"device": "cpu", "name": "CPU", "count": 0, "mem_gb": 0}
if torch.cuda.is_available():
    HW.update(
        device="cuda",
        count=torch.cuda.device_count(),
        name=torch.cuda.get_device_name(0),
        mem_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    )
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    HW.update(device="mps", name="Apple MPS", count=1)

DEVICE = torch.device(HW["device"])
print(f"  Device   : {HW['device']}")
print(f"  GPU      : {HW['name']}  ({HW['mem_gb']} GB)")
print(f"  Torch    : {torch.__version__}  TorchVision: {torchvision.__version__}  CUDA: {torch.version.cuda}")

if HW["device"] == "cpu":
    sys.stderr.write("\nWARNING: No CUDA-capable GPU detected. Training Faster R-CNN on CPU will be impractically slow.\n")

# Free any leftover GPU memory.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ---- Output layout ---------------------------------------------------
BASE_DIR = Path(".").resolve()
DATA_DIR = Path(LOCAL_DATASET_DIR).expanduser().resolve()
OUT_DIR  = BASE_DIR / "outputs_frcnn"   # separate from the YOLO outputs/

P = {
    "DATA":     DATA_DIR,
    "OUTPUT":   OUT_DIR,
    "CKPT":     OUT_DIR / "checkpoints",   # per-fold checkpoints live here
    "WEIGHTS":  OUT_DIR / "weights",       # final best.pt copies per fold
    "PLOTS":    OUT_DIR / "plots",
    "METRICS":  OUT_DIR / "metrics",
    "TILES":    BASE_DIR / "outputs" / "tiles",      # shared with YOLO nb
    "FOLDS":    BASE_DIR / "outputs" / "metrics",    # shared fold_splits.json
    "LOGS":     OUT_DIR / "logs",
}
for k, d in P.items():
    if k not in {"DATA", "TILES", "FOLDS"}:
        d.mkdir(parents=True, exist_ok=True)

# If the shared tile / fold dirs don't exist yet (notebook run standalone, not
# after the YOLO one), create them under outputs_frcnn instead.
if not P["TILES"].exists():
    P["TILES"] = OUT_DIR / "tiles"
    P["TILES"].mkdir(parents=True, exist_ok=True)
if not P["FOLDS"].exists():
    P["FOLDS"] = OUT_DIR / "metrics"
    P["FOLDS"].mkdir(parents=True, exist_ok=True)

print(f"  Dataset  : {P['DATA']}")
print(f"  Output   : {P['OUTPUT']}")
print(f"  Tiles    : {P['TILES']}")
print("[2.1] Done.")


In [ ]:
import sys
print(sys.executable)
import torch
print('torch', torch.__version__)
print('torch.version.cuda', torch.version.cuda)
print('cuda.is_available', torch.cuda.is_available())
print('cuda.device_count', torch.cuda.device_count() if torch.cuda.is_available() else 0)
print('cudnn.is_available', torch.backends.cudnn.is_available())


In [ ]:
!nvidia-smi


---
## 3. Dataset Acquisition (xView from Kaggle)

Downloads the xView dataset on first run; reuses the cache afterwards.


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  3.0  Auto-download the xView dataset from Kaggle (idempotent)
#
#  Source:  https://www.kaggle.com/datasets/hassanmojab/xview-dataset
#  Target:  ./dataset/xview/   (configurable via LOCAL_DATASET_DIR)
# ══════════════════════════════════════════════════════════════════
print("[3.0] Ensuring xView dataset is available ...")

DATA_DIR = Path(LOCAL_DATASET_DIR).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)


def _dataset_already_present(d: Path) -> bool:
    """True if d already holds a GeoJSON annotation + at least one image."""
    if not d.exists():
        return False
    try:
        has_geojson = next(d.rglob("*.geojson"), None) is not None
    except Exception:
        has_geojson = False
    try:
        has_images = (
            next(d.rglob("*.tif"), None) is not None
            or next(d.rglob("*.png"), None) is not None
            or next(d.rglob("*.jpg"), None) is not None
            or next(d.rglob("*.jpeg"), None) is not None
        )
    except Exception:
        has_images = False
    return has_geojson and has_images


def _ensure_kaggle_credentials():
    """Locate kaggle.json and copy it into ~/.kaggle/ with 0600 perms."""
    home_creds = Path.home() / ".kaggle" / "kaggle.json"
    if home_creds.exists():
        try:
            os.chmod(home_creds, 0o600)
        except Exception:
            pass
        return home_creds

    project_creds = Path("kaggle.json").resolve()
    if project_creds.exists():
        home_creds.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(project_creds, home_creds)
        try:
            os.chmod(home_creds, 0o600)
        except Exception:
            pass
        return home_creds

    raise FileNotFoundError(
        "Kaggle credentials not found. Download kaggle.json from "
        "https://www.kaggle.com/settings -> 'Create New API Token' and place it "
        "at either ./kaggle.json (project root) or ~/.kaggle/kaggle.json."
    )


if _dataset_already_present(DATA_DIR):
    print(f"  Dataset already present at {DATA_DIR} - skipping download.")
else:
    print(f"  Downloading {KAGGLE_DATASET!r} -> {DATA_DIR}")
    print("  (this is large; the download/unzip can take a while on first run)")

    _ensure_kaggle_credentials()
    _pip("kaggle", "kaggle")

    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files(
        KAGGLE_DATASET, path=str(DATA_DIR), unzip=True, quiet=False,
    )
    # Kaggle sometimes leaves the zip behind even with unzip=True.
    for z in DATA_DIR.glob("*.zip"):
        try: z.unlink()
        except Exception: pass
    print(f"  Done.  Dataset is now at {DATA_DIR}")

print("[3.0] Done.")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  3.1  Locate the local xView dataset
# ══════════════════════════════════════════════════════════════════
print("[3.1] Locating local dataset ...")

def safe_list_dirs(root: Path):
    try:
        return [p for p in root.rglob("*") if p.is_dir()]
    except Exception:
        return []

def find_xview(search_root: Path):
    """Recursively look for an images folder + a GeoJSON / labels folder."""
    empty = {"valid": False, "geojson": None, "images_dir": None, "labels_dir": None}
    if not search_root.exists():
        return empty

    all_dirs = [search_root] + safe_list_dirs(search_root)

    # ---- GeoJSON: prefer files that look like xView train annotations ----
    geo = None
    for pattern in ["*train*.geojson", "*xview*.geojson", "*xView*.geojson",
                    "*.geojson", "*.json"]:
        for p in search_root.rglob(pattern):
            if p.is_file():
                geo = p
                break
        if geo is not None:
            break

    # ---- Image folder ----------------------------------------------------
    img = None
    preferred_img = {"train_images", "images", "train", "imgs", "image", "jpegimages"}
    for d in all_dirs:
        try:
            if d.name.lower() in preferred_img:
                if any(d.glob("*.tif")) or any(d.glob("*.png")) \
                   or any(d.glob("*.jpg")) or any(d.glob("*.jpeg")):
                    img = d
                    break
        except Exception:
            pass
    if img is None:
        for d in all_dirs:
            try:
                if any(d.glob("*.tif")) or any(d.glob("*.png")) \
                   or any(d.glob("*.jpg")) or any(d.glob("*.jpeg")):
                    img = d
                    break
            except Exception:
                pass

    # ---- Labels folder (txt / geojson) -----------------------------------
    lbl = None
    preferred_lbl = {"train_labels", "labels", "label", "annotations", "annots", "anno"}
    for d in all_dirs:
        try:
            if d.name.lower() in preferred_lbl:
                if any(d.glob("*.txt")) or any(d.glob("*.geojson")) or any(d.glob("*.json")):
                    lbl = d
                    break
        except Exception:
            pass
    if lbl is None:
        for d in all_dirs:
            try:
                if any(d.glob("*.txt")) or any(d.glob("*.geojson")) or any(d.glob("*.json")):
                    lbl = d
                    break
            except Exception:
                pass
    if geo is not None and lbl is None:
        lbl = geo.parent

    valid = (img is not None) and ((geo is not None) or (lbl is not None))
    return {"valid": valid, "geojson": geo, "images_dir": img, "labels_dir": lbl}


DS_INFO = find_xview(P["DATA"])
if not DS_INFO["valid"]:
    raise FileNotFoundError(
        f"\n  xView dataset not found inside: {P['DATA']}\n"
        f"  Place the dataset there (must include a GeoJSON file and an\n"
        f"  images folder such as 'train_images/'), or set LOCAL_DATASET_DIR\n"
        f"  in cell 1.1 to the correct path."
    )

IMAGES_DIR   = DS_INFO["images_dir"]
GEOJSON_PATH = DS_INFO["geojson"]
LABELS_DIR   = DS_INFO["labels_dir"]

if GEOJSON_PATH is not None:
    ANNO_TYPE = "geojson"
elif LABELS_DIR is not None and list(LABELS_DIR.glob("*.geojson")):
    GEOJSON_PATH = list(LABELS_DIR.glob("*.geojson"))[0]
    ANNO_TYPE = "geojson"
else:
    ANNO_TYPE = "txt"

print(f"  DATA_DIR  : {P['DATA']}")
print(f"  Images    : {IMAGES_DIR}")
print(f"  Labels    : {LABELS_DIR}")
print(f"  Anno type : {ANNO_TYPE}")
print(f"  GeoJSON   : {GEOJSON_PATH}")
print("[3.1] Done.")


---
## 4. Annotation Loading & Class Selection

Parses the xView GeoJSON, then narrows to the four "small object" vehicle
classes (Table 3): Small Car, Passenger Vehicle, Pickup Truck, Utility Truck.


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  4.1  Load annotations
# ══════════════════════════════════════════════════════════════════
print("[4.1] Loading annotations ...")
t0 = time.time()

if ANNO_TYPE == "geojson":
    with open(GEOJSON_PATH) as f:
        raw = json.load(f)
    print(f"  GeoJSON features: {len(raw['features']):,}")
    records = []
    for feat in tqdm(raw["features"], desc="  Parsing"):
        props = feat.get("properties", {})
        coords = props.get("bounds_imcoords", "")
        if not coords:
            continue
        try:
            x1, y1, x2, y2 = (float(v) for v in coords.split(","))
        except Exception:
            continue
        w, h = abs(x2 - x1), abs(y2 - y1)
        records.append({
            "image_id": props.get("image_id", ""),
            "class_id": int(props.get("type_id", -1)),
            "px_x1": min(x1, x2), "px_y1": min(y1, y2),
            "px_x2": max(x1, x2), "px_y2": max(y1, y2),
            "px_w": w, "px_h": h, "px_area": w * h,
        })
    del raw
    bbox_df = pd.DataFrame(records)
else:
    raise NotImplementedError(f"TXT label loading not implemented (ANNO_TYPE={ANNO_TYPE})")

print(f"  Boxes: {len(bbox_df):,}  Images: {bbox_df['image_id'].nunique()}")
print(f"  Time : {time.time() - t0:.1f}s")
print("[4.1] Done.")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  4.2  Target class selection (Paper Table 3)
# ══════════════════════════════════════════════════════════════════
print("[4.2] Selecting target classes ...")

TARGET_CLS = {18: "Small Car", 17: "Passenger Vehicle",
              20: "Pickup Truck", 21: "Utility Truck"}
# In Faster R-CNN, class 0 is reserved for background, so foreground classes
# start at 1.
CLS_TO_IDX = {18: 1, 17: 2, 20: 3, 21: 4}
IDX_TO_NAME = {0: "background", 1: "Small_Car", 2: "Passenger_Vehicle",
               3: "Pickup_Truck", 4: "Utility_Truck"}
NUM_CLASSES = 5   # 4 foreground + 1 background

bbox_df["is_target"] = bbox_df["class_id"].isin(TARGET_CLS)
small_df = bbox_df[
    bbox_df["is_target"]
    & (bbox_df["px_area"] >= MIN_OBJ_PX)
    & (bbox_df["px_area"] <= MAX_OBJ_PX)
].copy()

for cid, cn in TARGET_CLS.items():
    s = small_df[small_df["class_id"] == cid]
    if len(s):
        print(f"  {cn:22s} ID={cid:2d}  {len(s):>8,} objs  "
              f"{s['image_id'].nunique():>4} imgs  "
              f"area {s['px_area'].min():.0f}-{s['px_area'].max():.0f}")
print(f"  Total: {len(small_df):,} objects across {small_df['image_id'].nunique()} images")
print("[4.2] Done.")


---
## 5. Exploratory Data Analysis

Reproduces Figure 3 (object-size + aspect-ratio distributions) and Table 3
(per-class summary). These motivate the choice of anchor size range
(10–90 px, slightly rectangular aspect).


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  5.1  Figure 3 + Table 3
# ══════════════════════════════════════════════════════════════════
print("[5.1] Plotting distributions ...")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
bins_s = np.logspace(0, np.log10(100_000), 20)
axes[0].hist(bbox_df["px_area"].clip(upper=100_000), bins=bins_s,
             alpha=.7, color="steelblue", edgecolor="w", label="All")
axes[0].hist(small_df["px_area"].clip(upper=100_000), bins=bins_s,
             alpha=.7, color="indianred", edgecolor="w", label="Target")
axes[0].axvline(1000, color="red", ls="--", alpha=.6)
axes[0].set_xscale("log")
axes[0].set_xlabel("Area (px)")
axes[0].set_title("(a) Size Distribution")
axes[0].legend()

short = np.minimum(small_df["px_w"], small_df["px_h"])
long_ = np.maximum(small_df["px_w"], small_df["px_h"])
ar = (short / long_.replace(0, np.nan)).dropna()
axes[1].hist(ar, bins=np.arange(0, 1.05, .1), color="indianred", edgecolor="w")
axes[1].set_xlabel("Aspect Ratio (short / long)")
axes[1].set_title("(b) Aspect Ratio")

plt.tight_layout()
plt.savefig(P["PLOTS"] / "fig3.png", dpi=150, bbox_inches="tight")
plt.show()

# Table 3
rows = []
for cid, cn in TARGET_CLS.items():
    s = small_df[small_df["class_id"] == cid]
    if len(s):
        rows.append({
            "Class": cn,
            "Images": s["image_id"].nunique(),
            "Objects": len(s),
            "Min": int(s["px_area"].min()),
            "Median": int(s["px_area"].median()),
            "Max": int(s["px_area"].max()),
        })
t3 = pd.DataFrame(rows)
print(t3.to_string(index=False))
t3.to_csv(P["METRICS"] / "table3.csv", index=False)
print("[5.1] Done.")


---
## 6. Image Tiling (512 × 512, Paper Section 4.1)

xView images are too large to feed to the network directly. We tile them
into 512 × 512 PNGs with YOLO-format labels (cx, cy, w, h normalised). The
cache survives between runs and is shared with the YOLOv11 notebook (same
`outputs/tiles/` directory), so if you've already tiled there, this is a
no-op.


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  6.1  Tiling pipeline (RAM-friendly, iterative save, cache-aware)
#
#  This notebook needs Faster-R-CNN-format labels (class ids 1..4, with
#  0 reserved for background). To stay compatible with the YOLO notebook
#  (which writes labels with class ids 0..3 into outputs/tiles/labels/),
#  we write FRCNN-format labels to a SEPARATE directory:
#       outputs/tiles/labels_frcnn/
#
#  Three possible starting states are handled:
#    (a) No cache at all              → tile from scratch, write images +
#                                       FRCNN labels (and YOLO labels too,
#                                       so a later YOLO run can reuse them).
#    (b) Tile images cached, no FRCNN → convert existing YOLO labels to
#        labels yet                     FRCNN labels (no re-tiling needed).
#    (c) Both image cache and          → just load the manifest and move on.
#        FRCNN labels cached
# ══════════════════════════════════════════════════════════════════
print("[6.1] Tiling ...")
t0 = time.time()

# ---- Path setup ----------------------------------------------------
img_out       = P["TILES"] / "images"
lbl_yolo_out  = P["TILES"] / "labels"          # YOLO format (cls 0..3)
lbl_frcnn_out = P["TILES"] / "labels_frcnn"    # FRCNN format (cls 1..4)
mf_csv        = P["TILES"] / "manifest.csv"
done_src_txt  = P["TILES"] / "_done_src.txt"
frcnn_sig     = lbl_frcnn_out / "_signature.json"

for d in (img_out, lbl_yolo_out, lbl_frcnn_out):
    d.mkdir(parents=True, exist_ok=True)

if FORCE_RETILE:
    print("  FORCE_RETILE=True — clearing tile cache ...")
    shutil.rmtree(P["TILES"], ignore_errors=True)
    for d in (img_out, lbl_yolo_out, lbl_frcnn_out):
        d.mkdir(parents=True, exist_ok=True)

# ---- Class-id mappings ---------------------------------------------
# CLS_TO_IDX maps xView class_id -> FRCNN class id (1..4).
# YOLO ids are simply FRCNN id - 1 (i.e. 0..3).
def xview_to_frcnn(cid):     return CLS_TO_IDX.get(cid, 1)
def xview_to_yolo(cid):      return CLS_TO_IDX.get(cid, 1) - 1

# ---- Helper functions ----------------------------------------------
def tile_pos(h, w, ts=512):
    """Return (y, x) origins covering an image with ~uniform overlap."""
    nr, nc = int(np.ceil(h / ts)), int(np.ceil(w / ts))
    out = []
    for r in range(nr):
        for c in range(nc):
            y = r * (h - ts) // max(nr - 1, 1) if nr > 1 else 0
            x = c * (w - ts) // max(nc - 1, 1) if nc > 1 else 0
            out.append((min(y, max(0, h - ts)), min(x, max(0, w - ts))))
    return out

def clip_box(bx, ty, tx, ts, mf=0.2):
    """Clip a box to the tile, dropping it if <20% of its area survives."""
    x1 = max(bx["px_x1"] - tx, 0)
    y1 = max(bx["px_y1"] - ty, 0)
    x2 = min(bx["px_x2"] - tx, ts)
    y2 = min(bx["px_y2"] - ty, ts)
    cw, ch = x2 - x1, y2 - y1
    if cw <= 0 or ch <= 0:
        return None
    if bx["px_area"] > 0 and cw * ch / bx["px_area"] < mf:
        return None
    return {"x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "w": cw, "h": ch, "cid": bx["class_id"]}

# ---- Inspect existing cache ----------------------------------------
existing_records, done_srcs = [], set()
if mf_csv.exists():
    try:
        prev = pd.read_csv(mf_csv)
        existing_records = prev.to_dict("records")
        done_srcs = set(prev["src"].unique().tolist())
        print(f"  Cached manifest: {len(prev):,} tiles from {len(done_srcs)} source images")
    except Exception:
        existing_records, done_srcs = [], set()
if done_src_txt.exists():
    try:
        for line in done_src_txt.read_text().splitlines():
            line = line.strip()
            if line:
                done_srcs.add(line)
    except Exception:
        pass

have_frcnn_labels = (
    frcnn_sig.exists() and any(lbl_frcnn_out.glob("*.txt"))
)
have_yolo_labels  = any(lbl_yolo_out.glob("*.txt"))

target_imgs = list(small_df["image_id"].unique())
if DEBUG_MODE:
    target_imgs = target_imgs[:30]
todo_imgs = [i for i in target_imgs if i not in done_srcs]

# ---- Case (c): everything is cached → just go ----------------------
# ---- Case (b): images cached but FRCNN labels missing → convert ----
# ---- Case (a): tile from scratch                                    -
if todo_imgs:
    print(f"  Tiling {len(todo_imgs)} / {len(target_imgs)} images "
          f"(skipping {len(done_srcs)} cached) ...")
    boxes_by_img = {iid: g for iid, g in small_df.groupby("image_id")}
    new_records, processed = [], 0
    FLUSH_EVERY = 5

    for img_id in tqdm(todo_imgs, desc="  Tiling"):
        # Resolve image path (handle alternate extensions)
        ip = IMAGES_DIR / str(img_id)
        if not ip.exists():
            for ext in (".tif", ".tiff", ".png"):
                c = IMAGES_DIR / (str(img_id).split(".")[0] + ext)
                if c.exists():
                    ip = c
                    break
        if not ip.exists():
            continue

        img = cv2.imread(str(ip))
        if img is None:
            continue
        h, w = img.shape[:2]
        iboxes = boxes_by_img.get(img_id)
        if iboxes is None or len(iboxes) == 0:
            del img
            continue

        for idx, (ty, tx) in enumerate(tile_pos(h, w, TILE_SIZE)):
            tile = img[ty:ty + TILE_SIZE, tx:tx + TILE_SIZE]
            if tile.shape[0] < TILE_SIZE or tile.shape[1] < TILE_SIZE:
                pad = np.zeros((TILE_SIZE, TILE_SIZE, 3), dtype=np.uint8)
                pad[:tile.shape[0], :tile.shape[1]] = tile
                tile = pad
            tboxes = [cb for _, bx in iboxes.iterrows()
                      if (cb := clip_box(bx, ty, tx, TILE_SIZE)) is not None]
            if not tboxes:
                continue

            tn = f"{Path(img_id).stem}_t{idx:04d}"
            tile_img_path = img_out / f"{tn}.png"
            if not tile_img_path.exists():
                cv2.imwrite(str(tile_img_path), tile)

            # Build both label files: YOLO (0..3) for the YOLO nb,
            # FRCNN (1..4) for this nb.
            yolo_lines, frcnn_lines = [], []
            for b in tboxes:
                cx = (b["x1"] + b["x2"]) / 2 / TILE_SIZE
                cy = (b["y1"] + b["y2"]) / 2 / TILE_SIZE
                bw = b["w"] / TILE_SIZE
                bh = b["h"] / TILE_SIZE
                yolo_lines .append(f"{xview_to_yolo(b['cid'])} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                frcnn_lines.append(f"{xview_to_frcnn(b['cid'])} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

            # Don't clobber an existing YOLO label file written by the YOLO nb.
            yolo_path = lbl_yolo_out / f"{tn}.txt"
            if not yolo_path.exists():
                yolo_path.write_text("\n".join(yolo_lines))
            (lbl_frcnn_out / f"{tn}.txt").write_text("\n".join(frcnn_lines))
            new_records.append({"tile": tn, "src": img_id, "n_obj": len(tboxes)})

        del img
        done_srcs.add(img_id)
        processed += 1
        if processed % FLUSH_EVERY == 0:
            combined = pd.DataFrame(existing_records + new_records)
            combined.drop_duplicates(subset=["tile"], keep="last", inplace=True)
            combined.to_csv(mf_csv, index=False)
            done_src_txt.write_text("\n".join(sorted(done_srcs)))
            gc.collect()

    combined = pd.DataFrame(existing_records + new_records)
    combined.drop_duplicates(subset=["tile"], keep="last", inplace=True)
    combined.to_csv(mf_csv, index=False)
    done_src_txt.write_text("\n".join(sorted(done_srcs)))
    del boxes_by_img
    gc.collect()
elif not have_frcnn_labels and have_yolo_labels:
    # Case (b): tile images + YOLO labels exist (typically from running the
    # YOLO notebook first); generate FRCNN labels from them.
    YOLO_IDX_TO_XVIEW = {0: 18, 1: 17, 2: 20, 3: 21}
    yolo_files = list(lbl_yolo_out.glob("*.txt"))
    print(f"  Converting {len(yolo_files):,} cached YOLO labels → FRCNN labels ...")
    for ytxt in tqdm(yolo_files, desc="  Converting"):
        frcnn_path = lbl_frcnn_out / f"{ytxt.stem}.txt"
        if frcnn_path.exists():
            continue
        out_lines = []
        for ln in ytxt.read_text().strip().splitlines():
            parts = ln.split()
            if len(parts) != 5:
                continue
            yolo_cid = int(parts[0])
            xv_cid   = YOLO_IDX_TO_XVIEW.get(yolo_cid, 18)
            frcnn_cid = xview_to_frcnn(xv_cid)
            out_lines.append(f"{frcnn_cid} {parts[1]} {parts[2]} {parts[3]} {parts[4]}")
        frcnn_path.write_text("\n".join(out_lines))
else:
    print(f"  All {len(target_imgs)} source images already tiled with FRCNN labels — using cache")

# Mark FRCNN-label dir as ready
frcnn_sig.write_text(json.dumps({"format": "class cx cy w h, class ids 1..4 (0=background)"}))

manifest = pd.read_csv(mf_csv)
n_frcnn = len(list(lbl_frcnn_out.glob("*.txt")))
print(f"  Tiles    : {len(manifest):,}   Objects: {manifest['n_obj'].sum():,}")
print(f"  Labels   : {n_frcnn:,} FRCNN-format files in {lbl_frcnn_out}")
print(f"  Time     : {time.time() - t0:.1f}s")
print("[6.1] Done.")


---
## 7. 6-Fold Cross-Validation Splits

Generates (or reuses) a deterministic 6-fold split *at the source-image
level* — tiles from the same original xView image always land in the same
fold, preventing leakage. The split is cached in `fold_splits.json`; the
YOLOv11 notebook uses the same file, so both notebooks evaluate on
identical splits and the numbers from Table 13 are directly comparable.


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  7.1  Generate (or load) deterministic 6-fold split
# ══════════════════════════════════════════════════════════════════
print("[7.1] Generating / loading 6-fold CV splits ...")
from sklearn.model_selection import KFold

fold_splits_path = P["FOLDS"] / "fold_splits.json"

src_images = sorted(manifest["src"].unique().tolist())

manifest_signature = {
    "num_src_images": len(src_images),
    "num_tiles":      int(len(manifest)),
    "num_folds":      NUM_FOLDS,
    "seed":           SEED,
}

def _build_fold_splits():
    kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
    splits = {}
    for fold_idx, (train_src_idx, val_src_idx) in enumerate(kf.split(src_images)):
        train_srcs = {src_images[i] for i in train_src_idx}
        val_srcs   = {src_images[i] for i in val_src_idx}
        splits[fold_idx] = {
            "train": manifest.loc[manifest["src"].isin(train_srcs), "tile"].tolist(),
            "val":   manifest.loc[manifest["src"].isin(val_srcs),   "tile"].tolist(),
        }
    return splits

fold_splits = None
if fold_splits_path.exists() and not FORCE_RESPLIT:
    try:
        cached = json.loads(fold_splits_path.read_text())
        if "signature" not in cached or "splits" not in cached:
            print("  Old-schema fold_splits.json - regenerating")
        elif cached["signature"] != manifest_signature:
            print(f"  Split signature changed - regenerating")
        else:
            fold_splits = {int(k): v for k, v in cached["splits"].items()}
            print(f"  Loaded cached split from {fold_splits_path.name} "
                  f"({manifest_signature['num_src_images']} src imgs, "
                  f"{manifest_signature['num_tiles']} tiles)")
    except Exception as e:
        print(f"  Cached split unreadable ({e}) - regenerating")
        fold_splits = None

if fold_splits is None:
    print("  Building fresh deterministic split ...")
    fold_splits = _build_fold_splits()
    payload = {
        "signature": manifest_signature,
        "splits":    {str(k): v for k, v in fold_splits.items()},
    }
    fold_splits_path.write_text(json.dumps(payload, indent=2))
    print(f"  Saved split to {fold_splits_path}")

print(f"  Source images: {len(src_images)}")
print(f"  Total tiles  : {len(manifest)}")
for fold_idx in range(NUM_FOLDS):
    s = fold_splits[fold_idx]
    n_tr_src = manifest.loc[manifest["tile"].isin(s["train"]), "src"].nunique()
    n_vl_src = manifest.loc[manifest["tile"].isin(s["val"]),   "src"].nunique()
    print(f"  Fold {fold_idx}: train={len(s['train']):>5} tiles  "
          f"val={len(s['val']):>5} tiles  "
          f"[src: {n_tr_src} / {n_vl_src}]")
print("[7.1] Done.")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  7.2  Training budget summary
# ══════════════════════════════════════════════════════════════════
print("[7.2] Training budget ...")

n_train_tiles    = len(fold_splits[0]["train"])
eff_batch        = BATCH_SIZE * GRAD_ACCUM
iters_per_epoch  = int(np.ceil(n_train_tiles / eff_batch))
paper_target_it  = 10_000        # Paper Section 4.1
paper_eq_epochs  = paper_target_it / max(iters_per_epoch, 1)

print(f"  Max epochs        : {EPOCHS}    (early stopping kicks in much earlier)")
print(f"  Patience          : {PATIENCE} epochs    Min Δ AP50:95: {MIN_DELTA}")
print(f"  Train tiles/fold  : {n_train_tiles:,}")
print(f"  Micro batch       : {BATCH_SIZE}")
print(f"  Grad accum steps  : {GRAD_ACCUM}")
print(f"  Effective batch   : {eff_batch}")
print(f"  Iters / epoch     : {iters_per_epoch:,}")
print(f"  Save every        : {CHECKPOINT_EVERY_N_STEPS} optimizer steps "
      f"({iters_per_epoch // max(CHECKPOINT_EVERY_N_STEPS, 1)} saves/epoch)")
print(f"  ≈ Max total iters : {iters_per_epoch * EPOCHS:,}   "
      f"(paper used {paper_target_it:,} = ~{paper_eq_epochs:.1f} epochs here)")
print()
print("  NOTE: With early stopping + a high EPOCHS ceiling, training keeps")
print("        going as long as the model improves and stops cleanly when it")
print("        plateaus. In practice, expect convergence around 25-45 epochs.")
print("[7.2] Done.")


---
## 8. Faster R-CNN: Model, Dataset, and Training Loop

### Model — paper Tables 5, 8, 12
- **Backbone**: ResNet-50 + FPN (`resnet_fpn_backbone`), ImageNet pre-trained
- **Anchor generator** (Section 4.2): sizes spanning **10–90 px** across the
  5 FPN levels (P2..P6, strides 4/8/16/32/64), 6 aspect ratios = the paper\'s
  3 short/long ratios + their reciprocals (vehicles in satellite imagery are
  arbitrarily rotated)
- **RPN region-proposal IoU threshold** (Table 8): **0.5** — the paper found
  this works best for small objects (vs torchvision default 0.7)
- **Frozen backbone** (Table 12): first 2 ResNet-50 stages frozen (~49 conv
  layers, matching the paper)

### Training loop — fully resume-safe
- **SGD** with the paper\'s exact hyperparameters (Table 5)
- **AMP** (`torch.cuda.amp`) for half-precision memory savings
- **Gradient accumulation**: micro-batch 2 × 8 = effective batch 16 (paper)
- **Per-epoch validation** with COCO-style mAP via `torchmetrics`
- **Intra-epoch checkpointing**: every `CHECKPOINT_EVERY_N_STEPS` optimizer
  steps the full state (model + optimizer + AMP scaler + history + patience
  counter) is atomically written to `last.pt`. A kernel crash, OS reboot, or
  power loss never costs more than ~`N × eff_batch` tiles of training.
- **End-of-epoch checkpointing**: same file is overwritten with
  `completed_epoch = epoch`, so resume picks up at the next epoch cleanly.
- **Best-so-far checkpoint** (`best.pt`): saved only when AP50:95 improves by
  more than `MIN_DELTA` percentage points — final eval uses these weights.
- **Early stopping**: stops after `PATIENCE` epochs with no qualifying
  improvement; the patience counter itself is persisted in the checkpoint,
  so resume doesn\'t reset the clock.
- **Per-fold status tracking** (`training_state.json`):
  pending → running → completed | failed | interrupted, every transition
  written atomically to disk.


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  8.1  Dataset class (reads 512×512 PNGs + YOLO-format FRCNN labels)
# ══════════════════════════════════════════════════════════════════
print("[8.1] Defining Dataset class ...")

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF


class XViewTileDataset(Dataset):
    """Reads tile PNGs and converts YOLO-format labels to Faster R-CNN
    target dicts (boxes in xyxy + labels in 1..NUM_CLASSES-1)."""

    def __init__(self, tile_names, images_dir, labels_dir, tile_size=512,
                 flip_prob=0.0):
        self.tile_names = list(tile_names)
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.tile_size  = tile_size
        self.flip_prob  = flip_prob
        # Filter to tiles that actually exist on disk (defensive).
        self.tile_names = [
            t for t in self.tile_names
            if (self.images_dir / f"{t}.png").exists()
        ]

    def __len__(self):
        return len(self.tile_names)

    def _load_target(self, tn):
        lp = self.labels_dir / f"{tn}.txt"
        boxes, labels = [], []
        if lp.exists():
            for ln in lp.read_text().strip().splitlines():
                parts = ln.split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, w, h = parts
                cls = int(cls)
                cx, cy, w, h = float(cx), float(cy), float(w), float(h)
                x1 = (cx - w / 2) * self.tile_size
                y1 = (cy - h / 2) * self.tile_size
                x2 = (cx + w / 2) * self.tile_size
                y2 = (cy + h / 2) * self.tile_size
                # Clip + drop degenerate boxes.
                x1 = max(0.0, min(x1, self.tile_size - 1))
                y1 = max(0.0, min(y1, self.tile_size - 1))
                x2 = max(0.0, min(x2, self.tile_size - 1))
                y2 = max(0.0, min(y2, self.tile_size - 1))
                if x2 - x1 < 1 or y2 - y1 < 1:
                    continue
                boxes.append([x1, y1, x2, y2])
                labels.append(cls)
        if not boxes:
            return (
                torch.zeros((0, 4), dtype=torch.float32),
                torch.zeros((0,), dtype=torch.int64),
            )
        return (
            torch.tensor(boxes, dtype=torch.float32),
            torch.tensor(labels, dtype=torch.int64),
        )

    def __getitem__(self, idx):
        tn = self.tile_names[idx]
        ip = self.images_dir / f"{tn}.png"
        img_bgr = cv2.imread(str(ip))
        if img_bgr is None:
            raise FileNotFoundError(f"Could not read tile image: {ip}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        boxes, labels = self._load_target(tn)

        # Horizontal flip augmentation (paper: p=0.5).
        if self.flip_prob > 0 and random.random() < self.flip_prob:
            img_rgb = img_rgb[:, ::-1, :].copy()
            if len(boxes):
                x1 = self.tile_size - boxes[:, 2]
                x2 = self.tile_size - boxes[:, 0]
                boxes = torch.stack([x1, boxes[:, 1], x2, boxes[:, 3]], dim=1)

        img_t = TF.to_tensor(img_rgb)  # HxWx3 uint8 -> 3xHxW float in [0,1]

        target = {
            "boxes":     boxes,
            "labels":    labels,
            "image_id":  torch.tensor([idx]),
            "area":      (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
                          if len(boxes) else torch.zeros((0,), dtype=torch.float32),
            "iscrowd":   torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return img_t, target


def collate_fn(batch):
    return tuple(zip(*batch))


# Image + FRCNN label directories.
TILE_IMG_DIR        = P["TILES"] / "images"
TILE_LBL_FRCNN_DIR  = P["TILES"] / "labels_frcnn"
print(f"  Tile images : {TILE_IMG_DIR}")
print(f"  FRCNN labels: {TILE_LBL_FRCNN_DIR}")
print("[8.1] Done.")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  8.2  Faster R-CNN model builder
# ══════════════════════════════════════════════════════════════════
print("[8.2] Defining model builder ...")

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models import resnet50, ResNet50_Weights


def build_anchor_generator(min_px, max_px, ratios):
    """Build an AnchorGenerator whose sizes span min_px..max_px across
    5 FPN levels (P2..P6 = strides 4, 8, 16, 32, 64)."""
    # Geometric progression so the final/coarsest level has size == max_px and
    # the first/finest level has size == min_px.
    sizes_per_level = np.geomspace(min_px, max_px, 5).round().astype(int).tolist()
    # AnchorGenerator wants one tuple of sizes per FPN level.
    sizes = tuple((int(s),) for s in sizes_per_level)
    aspect_ratios = (tuple(ratios),) * len(sizes)
    return AnchorGenerator(sizes=sizes, aspect_ratios=aspect_ratios)


def build_faster_rcnn(num_classes, anchor_min, anchor_max, anchor_ratios,
                      trainable_backbone_layers=3,
                      rpn_score_thresh=0.0,
                      box_score_thresh=0.05,
                      nms_iou_thresh=0.5):
    """Build a Faster R-CNN model with a ResNet-50 + FPN backbone and a
    custom anchor generator tuned for small objects."""
    # `trainable_backbone_layers` is in [0, 5]; default 3 freezes the first
    # two stages. Setting to 3 ≈ paper's 49 frozen conv layers in Table 12.
    backbone = resnet_fpn_backbone(
        backbone_name="resnet50",
        weights=ResNet50_Weights.IMAGENET1K_V1,
        trainable_layers=trainable_backbone_layers,
    )
    anchor_gen = build_anchor_generator(anchor_min, anchor_max, anchor_ratios)

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_gen,
        rpn_score_thresh=rpn_score_thresh,
        box_score_thresh=box_score_thresh,
        box_nms_thresh=nms_iou_thresh,
        # The Faster R-CNN paper's region-proposal threshold is the IoU at
        # which RPN proposals are labelled as positives (Section 4.3 of the
        # empirical paper). The default is 0.7; the empirical paper found
        # 0.5 worked best for small objects (Table 8).
        rpn_fg_iou_thresh=0.5,
        rpn_bg_iou_thresh=0.3,
        # Inputs are already 512×512 RGB in [0,1]; skip the default 800-side resize.
        min_size=512,
        max_size=512,
    )
    return model, anchor_gen


# Smoke-test the model on dummy input so any anchor-generator misconfig
# surfaces here, not 1000 iterations into training.
_test_model, _anchor_gen = build_faster_rcnn(
    NUM_CLASSES, ANCHOR_MIN_PX, ANCHOR_MAX_PX, ANCHOR_ASPECTS,
    trainable_backbone_layers=FROZEN_BACKBONE_LAYERS,
)
n_params = sum(p.numel() for p in _test_model.parameters()) / 1e6
print(f"  Model       : Faster R-CNN (ResNet-50 + FPN)")
print(f"  Params      : {n_params:.2f}M  (paper: 41.29M)")
print(f"  Anchor sizes: {_anchor_gen.sizes}")
print(f"  Aspect rats : {_anchor_gen.aspect_ratios[0]}")
print(f"  Frozen lyrs : {FROZEN_BACKBONE_LAYERS} stages (~49 conv layers; paper Table 12)")
del _test_model, _anchor_gen
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("[8.2] Done.")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  8.3  Training + evaluation routines
#
#  train_one_epoch() now supports INTRA-EPOCH CHECKPOINTING via a
#  caller-supplied callback. Every CHECKPOINT_EVERY_N_STEPS optimizer
#  steps, the callback is invoked so the outer fold-training loop can
#  persist the current state (weights + optimizer + scaler + history +
#  patience counter). If the process is killed mid-epoch and restarted,
#  the next run loads that checkpoint and continues with already-advanced
#  weights, so at most CHECKPOINT_EVERY_N_STEPS × eff_batch tiles of
#  training are repeated.
# ══════════════════════════════════════════════════════════════════
print("[8.3] Defining training routines ...")

from torch.optim import SGD
from torch.optim.lr_scheduler import LambdaLR


def _make_loaders(fold_idx):
    """Build the train/val DataLoaders for a given fold."""
    s = fold_splits[fold_idx]
    train_ds = XViewTileDataset(s["train"], TILE_IMG_DIR, TILE_LBL_FRCNN_DIR,
                                tile_size=TILE_SIZE, flip_prob=FLIP_PROB)
    val_ds   = XViewTileDataset(s["val"],   TILE_IMG_DIR, TILE_LBL_FRCNN_DIR,
                                tile_size=TILE_SIZE, flip_prob=0.0)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, collate_fn=collate_fn,
        pin_memory=torch.cuda.is_available(), drop_last=True,
        persistent_workers=(NUM_WORKERS > 0),
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, collate_fn=collate_fn,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )
    return train_ds, val_ds, train_loader, val_loader


def train_one_epoch(model, optimizer, loader, scaler, epoch, grad_accum,
                    checkpoint_callback=None, checkpoint_every_n_steps=0):
    """
    Run one epoch of training.

    Parameters
    ----------
    checkpoint_callback : callable or None
        Zero-arg function the outer loop uses to persist mid-epoch state.
        Called every `checkpoint_every_n_steps` optimizer steps.
    checkpoint_every_n_steps : int
        How many OPTIMIZER steps (not micro-batches) between callbacks.
        0 disables intra-epoch saves; only end-of-epoch saves happen.

    Returns
    -------
    dict
        Per-component running-average losses for the epoch.
    """
    model.train()
    running   = defaultdict(float)
    n_batches = len(loader)
    optimizer.zero_grad(set_to_none=True)
    optim_step_count = 0

    pbar = tqdm(enumerate(loader), total=n_batches,
                desc=f"    Epoch {epoch} train", leave=False)
    for step, (images, targets) in pbar:
        images  = [img.to(DEVICE, non_blocking=True) for img in images]
        targets = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()}
                   for t in targets]
        # Skip batches where every target is empty (torchvision throws).
        if all(len(t["boxes"]) == 0 for t in targets):
            continue

        with torch.cuda.amp.autocast(enabled=(AMP and torch.cuda.is_available())):
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values()) / grad_accum

        # Skip non-finite losses (rare; happens with degenerate anchors).
        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()

        is_step_boundary = ((step + 1) % grad_accum == 0
                            or (step + 1) == n_batches)
        if is_step_boundary:
            # Gradient clipping helps when LR is high relative to the small
            # losses produced by tiny objects.
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            optim_step_count += 1

            # ── Intra-epoch checkpoint ────────────────────────────────
            # Persist mid-epoch state every N optimizer steps. If the
            # process dies after this point, the next run will resume with
            # the already-applied gradient updates.
            if (checkpoint_callback is not None
                    and checkpoint_every_n_steps > 0
                    and optim_step_count % checkpoint_every_n_steps == 0):
                try:
                    checkpoint_callback()
                except Exception as _e:
                    # We never want the checkpoint write itself to break
                    # training. Print and continue.
                    print(f"    [WARN] intra-epoch checkpoint failed: {_e}")

        for k, v in loss_dict.items():
            running[k] += float(v.detach())
        running["loss"] += float(loss.detach()) * grad_accum

        if step % 25 == 0:
            pbar.set_postfix({
                "loss":  f"{running['loss'] / (step + 1):.3f}",
                "rpn_c": f"{running.get('loss_objectness', 0) / (step + 1):.3f}",
                "rpn_b": f"{running.get('loss_rpn_box_reg', 0) / (step + 1):.3f}",
                "roi_c": f"{running.get('loss_classifier', 0) / (step + 1):.3f}",
                "roi_b": f"{running.get('loss_box_reg', 0) / (step + 1):.3f}",
                "saves": optim_step_count // max(checkpoint_every_n_steps, 1)
                         if checkpoint_every_n_steps else 0,
            })

    return {k: v / max(n_batches, 1) for k, v in running.items()}


@torch.no_grad()
def evaluate(model, loader, conf_threshold=0.05):
    """Returns dict with AP50, AP50:95, precision@0.5, recall@0.5, F1@0.5."""
    model.eval()
    # COCO-style mAP via torchmetrics.
    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=[round(0.5 + 0.05 * i, 2) for i in range(10)],
        class_metrics=False,
    )

    # We also maintain matched TP/FP/FN counts at IoU=0.5 so we can compute
    # precision / recall / F1 the way Table 13 does.
    tp50 = fp50 = fn50 = 0

    def _box_iou_np(a, b):
        if len(a) == 0 or len(b) == 0:
            return np.zeros((len(a), len(b)), dtype=np.float32)
        ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
        bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
        ix1 = np.maximum(ax1, bx1); iy1 = np.maximum(ay1, by1)
        ix2 = np.minimum(ax2, bx2); iy2 = np.minimum(ay2, by2)
        iw = np.clip(ix2 - ix1, 0, None); ih = np.clip(iy2 - iy1, 0, None)
        inter  = iw * ih
        area_a = (ax2 - ax1) * (ay2 - ay1)
        area_b = (bx2 - bx1) * (by2 - by1)
        union  = area_a + area_b - inter
        return inter / np.maximum(union, 1e-9)

    for images, targets in tqdm(loader, desc="    Eval", leave=False):
        images = [img.to(DEVICE, non_blocking=True) for img in images]
        with torch.cuda.amp.autocast(enabled=(AMP and torch.cuda.is_available())):
            outputs = model(images)

        preds_for_metric, targets_for_metric = [], []
        for out, tgt in zip(outputs, targets):
            keep = out["scores"] >= conf_threshold
            pb = out["boxes"][keep].detach().cpu()
            ps = out["scores"][keep].detach().cpu()
            pl = out["labels"][keep].detach().cpu()
            preds_for_metric.append({"boxes": pb, "scores": ps, "labels": pl})
            targets_for_metric.append({
                "boxes":  tgt["boxes"].detach().cpu(),
                "labels": tgt["labels"].detach().cpu(),
            })

            # Matched TP/FP/FN at IoU >= 0.5.
            gt = tgt["boxes"].detach().cpu().numpy()
            pr = pb.numpy()
            if len(pr) == 0 and len(gt) == 0:
                continue
            if len(pr) == 0:
                fn50 += len(gt)
                continue
            if len(gt) == 0:
                fp50 += len(pr)
                continue
            order      = np.argsort(-ps.numpy())
            pr_sorted  = pr[order]
            ious       = _box_iou_np(pr_sorted, gt)
            matched_gt = set()
            tp_here    = 0
            for i in range(len(pr_sorted)):
                row = ious[i].copy()
                for g_idx in matched_gt:
                    row[g_idx] = -1
                bj = int(np.argmax(row))
                if row[bj] >= 0.5:
                    matched_gt.add(bj); tp_here += 1
            tp50 += tp_here
            fp50 += len(pr_sorted) - tp_here
            fn50 += len(gt) - tp_here

        metric.update(preds_for_metric, targets_for_metric)

    r         = metric.compute()
    precision = tp50 / max(tp50 + fp50, 1)
    recall    = tp50 / max(tp50 + fn50, 1)
    f1        = 2 * precision * recall / max(precision + recall, 1e-9)

    return {
        "AP50":      round(float(r["map_50"]) * 100, 2),
        "AP50_95":   round(float(r["map"])    * 100, 2),
        "Precision": round(precision * 100, 2),
        "Recall":    round(recall    * 100, 2),
        "F1":        round(f1        * 100, 2),
        "tp": int(tp50), "fp": int(fp50), "fn": int(fn50),
    }


print("[8.3] Done.")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  8.4  Per-fold training loop  (resume-safe, multi-fold, fully crash-tolerant)
#
#  Key design points:
#    1. CHECKPOINT FORMAT — a single .pt file holds model + optimizer + AMP
#       scaler + history + patience counter + best metric + the in-progress
#       epoch number. All writes are ATOMIC (write-to-.tmp then os.replace).
#    2. INTRA-EPOCH SAVES — train_one_epoch() calls _save_in_progress() every
#       CHECKPOINT_EVERY_N_STEPS optimizer steps. The saved checkpoint has
#       completed_epoch = (epoch - 1), so on resume the loop restarts the
#       current epoch from step 0 but with weights/optimizer state already
#       advanced. Worst case: ~N×eff_batch tiles of redone work.
#    3. END-OF-EPOCH SAVES — last.pt updated with completed_epoch = epoch.
#       best.pt updated only when AP50:95 improves by > MIN_DELTA.
#    4. EARLY STOPPING — patience counter (no_improve) is also persisted, so
#       resumes don\'t reset the early-stop clock.
#    5. FOLD STATUS — training_state.json tracks each fold\'s lifecycle:
#       pending → running → completed | failed | interrupted.
# ══════════════════════════════════════════════════════════════════
print("[8.4] Starting fold training loop ...")

STATE_PATH = P["METRICS"] / "training_state.json"


def _now():
    return datetime.now().isoformat(timespec="seconds")


# ── Training-state JSON (one global file per training run) ──────────
def load_state():
    if STATE_PATH.exists():
        try:
            return json.loads(STATE_PATH.read_text())
        except Exception as e:
            print(f"  training_state.json unreadable ({e}) - starting fresh")
    return {
        "version":    2,
        "num_folds":  NUM_FOLDS,
        "seed":       SEED,
        "created_at": _now(),
        "updated_at": _now(),
        "folds":      {str(i): {"status": "pending"} for i in range(NUM_FOLDS)},
    }


def save_state(state):
    """Atomic write: dump to .tmp then os.replace."""
    state["updated_at"] = _now()
    tmp = STATE_PATH.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(state, indent=2))
    os.replace(str(tmp), str(STATE_PATH))


def update_fold(state, fi, **kw):
    f = state["folds"].setdefault(str(fi), {})
    f.update(kw)
    save_state(state)


state = load_state()
if state.get("num_folds") != NUM_FOLDS or state.get("seed") != SEED:
    print("  num_folds/seed changed — resetting training_state.json")
    state = {
        "version":    2,
        "num_folds":  NUM_FOLDS,
        "seed":       SEED,
        "created_at": _now(),
        "updated_at": _now(),
        "folds":      {str(i): {"status": "pending"} for i in range(NUM_FOLDS)},
    }
    save_state(state)


# ── Per-fold helpers ────────────────────────────────────────────────
def fold_dir(fold_idx):
    d = P["CKPT"] / f"frcnn_fold{fold_idx}"
    d.mkdir(parents=True, exist_ok=True)
    return d


def fold_is_complete(fold_idx):
    """A fold is considered finished if it has both a best.pt and a CSV."""
    return (
        (P["METRICS"] / f"fold{fold_idx}_results.csv").exists()
        and (fold_dir(fold_idx) / "best.pt").exists()
    )


def save_checkpoint(path, model, optimizer, scaler, epoch,
                    best_metric, history, hp,
                    completed_epoch=None, no_improve=0):
    """
    Atomic save of the full training state.

    `epoch`            = the epoch currently being trained
    `completed_epoch`  = last fully-finished epoch (None if none)
    `no_improve`       = patience counter (epochs since last AP50:95 improvement)
    """
    tmp = path.with_suffix(".pt.tmp")
    torch.save({
        "model":           model.state_dict(),
        "optimizer":       optimizer.state_dict(),
        "scaler":          scaler.state_dict(),
        "epoch":           epoch,
        "completed_epoch": completed_epoch,
        "no_improve":      no_improve,
        "best_metric":     best_metric,
        "history":         history,
        "hp":              hp,
        "saved_at":        _now(),
    }, tmp)
    os.replace(str(tmp), str(path))


def load_checkpoint(path, model, optimizer=None, scaler=None):
    ck = torch.load(str(path), map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model"])
    if optimizer is not None and "optimizer" in ck:
        optimizer.load_state_dict(ck["optimizer"])
    if scaler is not None and "scaler" in ck:
        scaler.load_state_dict(ck["scaler"])
    return ck


# ── Main per-fold training routine ──────────────────────────────────
def train_fold(fold_idx, state):
    fd          = fold_dir(fold_idx)
    last_pt     = fd / "last.pt"
    best_pt     = fd / "best.pt"
    history_csv = fd / "history.csv"

    train_ds, val_ds, train_loader, val_loader = _make_loaders(fold_idx)
    print(f"    Fold {fold_idx}: {len(train_ds)} train, {len(val_ds)} val tiles")

    # Build model + optimizer + scaler from scratch every time. If we\'re
    # resuming, the checkpoint state-dicts will overwrite them below.
    model, _ = build_faster_rcnn(
        NUM_CLASSES, ANCHOR_MIN_PX, ANCHOR_MAX_PX, ANCHOR_ASPECTS,
        trainable_backbone_layers=FROZEN_BACKBONE_LAYERS,
    )
    model.to(DEVICE)

    params    = [p for p in model.parameters() if p.requires_grad]
    optimizer = SGD(params, lr=LEARNING_RATE,
                    momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scaler    = torch.cuda.amp.GradScaler(enabled=(AMP and torch.cuda.is_available()))

    # ── Default initial state (when no checkpoint exists) ────────────
    start_epoch = 1
    best_metric = -1.0
    no_improve  = 0
    history     = []
    final_metrics = None   # populated either after training or from best.pt
    hp_snapshot = {
        "lr": LEARNING_RATE, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
        "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
        "anchor_min_px": ANCHOR_MIN_PX, "anchor_max_px": ANCHOR_MAX_PX,
        "anchor_aspects": ANCHOR_ASPECTS,
        "frozen_backbone_layers": FROZEN_BACKBONE_LAYERS,
        "epochs_max": EPOCHS, "patience": PATIENCE, "min_delta": MIN_DELTA,
        "checkpoint_every_n_steps": CHECKPOINT_EVERY_N_STEPS,
        "seed": SEED + fold_idx,
    }

    # ── Resume from last.pt if available ─────────────────────────────
    if RESUME_TRAINING and last_pt.exists():
        try:
            ck = load_checkpoint(last_pt, model, optimizer, scaler)
            best_metric = float(ck.get("best_metric", -1.0))
            history     = list(ck.get("history", []))
            no_improve  = int(ck.get("no_improve", 0))
            ce          = ck.get("completed_epoch")
            saved_epoch = int(ck.get("epoch", 0))
            if ce is not None:
                start_epoch = int(ce) + 1
                tag = (f"in-progress epoch {saved_epoch}"
                       if saved_epoch > int(ce) else
                       f"finished epoch {ce}")
            else:
                # Old-format checkpoint — treat saved epoch as completed.
                start_epoch = saved_epoch + 1
                tag = f"finished epoch {saved_epoch} (legacy ckpt)"
            print(f"    Resuming from {last_pt.name}: {tag} → "
                  f"starting epoch {start_epoch}  "
                  f"(best AP50:95 so far = {best_metric:.2f}, "
                  f"no_improve={no_improve})")
        except Exception as e:
            print(f"    Could not resume from {last_pt.name} ({e}) — starting fresh")
            start_epoch, best_metric, history, no_improve = 1, -1.0, [], 0

    update_fold(state, fold_idx,
                status="running", started_at=_now(),
                start_epoch=start_epoch)

    # If patience was already exhausted before the crash, just finalize.
    if start_epoch > EPOCHS:
        print(f"    All {EPOCHS} epochs already completed for fold {fold_idx} — finalizing.")
    elif no_improve >= PATIENCE:
        print(f"    Patience already exhausted (no_improve={no_improve} ≥ {PATIENCE}) — finalizing.")
    else:
        # ── Main epoch loop ───────────────────────────────────────────
        current_epoch_state = {"epoch": start_epoch}

        def _save_in_progress():
            """Mid-epoch save. Note: completed_epoch = current epoch - 1."""
            save_checkpoint(last_pt, model, optimizer, scaler,
                            epoch=current_epoch_state["epoch"],
                            best_metric=best_metric,
                            history=history,
                            hp=hp_snapshot,
                            completed_epoch=current_epoch_state["epoch"] - 1,
                            no_improve=no_improve)

        for epoch in range(start_epoch, EPOCHS + 1):
            current_epoch_state["epoch"] = epoch
            t_ep = time.time()
            torch.manual_seed(SEED + fold_idx + epoch)
            random.seed(SEED + fold_idx + epoch)

            try:
                train_losses = train_one_epoch(
                    model, optimizer, train_loader, scaler, epoch, GRAD_ACCUM,
                    checkpoint_callback=_save_in_progress,
                    checkpoint_every_n_steps=CHECKPOINT_EVERY_N_STEPS,
                )
            except torch.cuda.OutOfMemoryError as e:
                print(f"    OOM at epoch {epoch}: {e}")
                print("    Try lowering BATCH_SIZE (and bumping GRAD_ACCUM to compensate)")
                gc.collect(); torch.cuda.empty_cache()
                raise

            eval_metrics = evaluate(model, val_loader, conf_threshold=BOX_SCORE_THRESH)
            final_metrics = eval_metrics  # remember latest in case best.pt is missing

            # ── Did we improve by more than MIN_DELTA? ────────────────
            improved = eval_metrics["AP50_95"] > (best_metric + MIN_DELTA)
            if improved:
                best_metric = eval_metrics["AP50_95"]
                no_improve  = 0
                save_checkpoint(best_pt, model, optimizer, scaler, epoch,
                                best_metric, history, hp_snapshot,
                                completed_epoch=epoch,
                                no_improve=no_improve)
            else:
                no_improve += 1

            # Append history AFTER deciding `improved` so we record the right epoch.
            row = {"epoch": epoch, **train_losses, **eval_metrics,
                   "epoch_time_min": round((time.time() - t_ep) / 60, 2),
                   "best_so_far_AP50_95": best_metric,
                   "no_improve": no_improve}
            history.append(row)

            # End-of-epoch save (last.pt with completed_epoch = epoch).
            save_checkpoint(last_pt, model, optimizer, scaler, epoch,
                            best_metric, history, hp_snapshot,
                            completed_epoch=epoch,
                            no_improve=no_improve)
            pd.DataFrame(history).to_csv(history_csv, index=False)

            update_fold(state, fold_idx,
                        last_epoch=epoch,
                        best_AP50_95=best_metric,
                        last_metrics=eval_metrics,
                        no_improve=no_improve)

            print(f"    Epoch {epoch:>3d}/{EPOCHS}  "
                  f"loss={train_losses['loss']:.3f}  "
                  f"AP50={eval_metrics['AP50']:.2f}  "
                  f"AP50:95={eval_metrics['AP50_95']:.2f}  "
                  f"F1={eval_metrics['F1']:.2f}  "
                  f"{'★ new best' if improved else '          '}  "
                  f"[{row['epoch_time_min']:.1f}m]  "
                  f"no_improve={no_improve}/{PATIENCE}")

            if no_improve >= PATIENCE:
                print(f"    ⏹ Early stopping triggered "
                      f"(no AP50:95 improvement > {MIN_DELTA} for {PATIENCE} epochs).")
                break

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # ── Final eval using BEST checkpoint ───────────────────────────────
    if best_pt.exists():
        load_checkpoint(best_pt, model)
        final_metrics = evaluate(model, val_loader, conf_threshold=BOX_SCORE_THRESH)
    elif final_metrics is None:
        # Truly nothing to evaluate (no epochs ran and no best.pt).
        final_metrics = {"AP50": 0.0, "AP50_95": 0.0, "Precision": 0.0,
                         "Recall": 0.0, "F1": 0.0, "tp": 0, "fp": 0, "fn": 0}

    # Persist a clean per-fold CSV in the standard location.
    out_row = {"fold": fold_idx, **final_metrics}
    pd.DataFrame([out_row]).to_csv(
        P["METRICS"] / f"fold{fold_idx}_results.csv", index=False)
    if best_pt.exists():
        shutil.copy2(best_pt, P["WEIGHTS"] / f"fold{fold_idx}_best.pt")

    update_fold(state, fold_idx,
                status="completed",
                completed_at=_now(),
                metrics=final_metrics,
                best_pt=str(best_pt))

    del model, optimizer, scaler, train_loader, val_loader, train_ds, val_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return final_metrics


# ── Decide which folds to run ──────────────────────────────────────
def _decide_folds_to_run():
    if isinstance(CURRENT_FOLD, int):
        candidates = [CURRENT_FOLD]
    elif CURRENT_FOLD == "all":
        candidates = list(range(NUM_FOLDS))
    else:
        candidates = [0]

    todo, skipped = [], []
    for fi in candidates:
        if fold_is_complete(fi):
            st = state["folds"].get(str(fi), {}).get("status")
            if st != "completed":
                update_fold(state, fi, status="completed", completed_at=_now(),
                            note="reconciled from on-disk artefacts")
            skipped.append(fi)
        else:
            todo.append(fi)
    return todo, skipped


folds_to_run, folds_skipped = _decide_folds_to_run()

print(f"  Folds requested  : {CURRENT_FOLD}")
print(f"  Folds skipped    : {folds_skipped}  (already completed)")
print(f"  Folds to run     : {folds_to_run}")
print(f"  Max epochs       : {EPOCHS}    Patience: {PATIENCE}    MinΔ: {MIN_DELTA}")
print(f"  Save every       : {CHECKPOINT_EVERY_N_STEPS} optimizer steps "
      "(plus end-of-epoch + best-so-far)")
print(f"  Effective batch  : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR / momentum    : {LEARNING_RATE:g} / {MOMENTUM}")

# Reload any previously-saved fold metrics from disk so the aggregated
# results table later in the notebook reflects completed folds even on
# a fresh kernel.
all_metrics = []
for fi in range(NUM_FOLDS):
    fp = P["METRICS"] / f"fold{fi}_results.csv"
    if fp.exists():
        try:
            all_metrics.append(pd.read_csv(fp).iloc[0].to_dict())
        except Exception:
            pass

try:
    for fi in folds_to_run:
        print(f"\n{'=' * 60}")
        print(f"  FOLD {fi}/{NUM_FOLDS - 1}")
        print(f"{'=' * 60}")

        if SKIP_TRAINING:
            print(f"    SKIP_TRAINING=True — skipping fold {fi}")
            continue

        t0 = time.time()
        try:
            met = train_fold(fi, state)
        except KeyboardInterrupt:
            update_fold(state, fi, status="interrupted", interrupted_at=_now())
            print("\n    Training interrupted. Checkpoints kept on disk for resume.")
            raise
        except Exception as e:
            update_fold(state, fi, status="failed",
                        error=f"train_fold: {e}", failed_at=_now())
            print(f"\n    Fold {fi} failed: {e}\n    Continuing with next fold.")
            import traceback; traceback.print_exc()
            continue
        print(f"    Total fold time: {(time.time() - t0) / 60:.1f} min")

        if met:
            all_metrics = [m for m in all_metrics if int(m.get("fold", -1)) != fi]
            all_metrics.append({"fold": fi, **met})
finally:
    try:
        save_state(state)
    except Exception as e:
        print(f"  WARNING: could not save training_state.json in finally: {e}")
    n_done = sum(1 for f in state["folds"].values() if f.get("status") == "completed")
    n_fail = sum(1 for f in state["folds"].values() if f.get("status") == "failed")
    n_intr = sum(1 for f in state["folds"].values() if f.get("status") == "interrupted")
    print(f"\n  Folds completed  : {n_done}/{NUM_FOLDS}")
    print(f"  Folds failed     : {n_fail}")
    print(f"  Folds interrupted: {n_intr}")
    print(f"  State file       : {STATE_PATH}")
    print("[8.4] Done.")


---
## 9. Aggregate Results Across Folds (Paper Table 13)


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  9.1  Aggregate fold results
# ══════════════════════════════════════════════════════════════════
print("[9.1] Aggregating results ...")

saved_results = []
for fi in range(NUM_FOLDS):
    fp = P["METRICS"] / f"fold{fi}_results.csv"
    if fp.exists():
        saved_results.append(pd.read_csv(fp).iloc[0].to_dict())

if saved_results:
    results_df = pd.DataFrame(saved_results)
    n_folds_done = len(results_df)
    print(f"  Results from {n_folds_done}/{NUM_FOLDS} folds:\n")
    print(results_df.to_string(index=False))

    metrics = ["AP50", "AP50_95", "Precision", "Recall", "F1"]
    paper = {"AP50": (61.65, 3.22), "AP50_95": (23.32, 1.30), "F1": (69.44, 2.07)}
    print(f"\n  {'Metric':12s}  {'Mean':>8s}  {'Std':>7s}  Paper (mean ± std)")
    print("  " + "-" * 55)
    for m in metrics:
        if m in results_df.columns:
            mean_v = results_df[m].mean()
            std_v  = results_df[m].std() if n_folds_done > 1 else 0
            p_str = ""
            if m in paper:
                pm, ps = paper[m]
                p_str = f"{pm:.2f} ± {ps:.2f}"
            print(f"  {m:12s}  {mean_v:8.2f}  {std_v:7.2f}  {p_str}")

    agg = {m: {"mean": float(results_df[m].mean()),
               "std":  float(results_df[m].std()) if n_folds_done > 1 else 0.0}
           for m in metrics if m in results_df.columns}
    agg["n_folds"] = n_folds_done
    with open(P["METRICS"] / "aggregated_results.json", "w") as f:
        json.dump(agg, f, indent=2)
    results_df.to_csv(P["METRICS"] / "all_folds_results.csv", index=False)
    print(f"\n  Saved: aggregated_results.json, all_folds_results.csv")

    if n_folds_done < NUM_FOLDS:
        print(f"\n  NOTE: {n_folds_done}/{NUM_FOLDS} folds done.")
        print(f"  Set CURRENT_FOLD = {n_folds_done} (or 'all') in cell 1.1 and re-run.")

    if DEBUG_MODE:
        print("\n  DEBUG results — not paper-comparable.")
else:
    print("  No fold results found yet. Train at least one fold first.")

print("[9.1] Done.")


---
## 10. Prediction Overlays + Training Curves


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  10.1  Prediction overlays from the most recently run fold
# ══════════════════════════════════════════════════════════════════
print("[10.1] Prediction overlays ...")

last_fold = folds_to_run[-1] if folds_to_run else (folds_skipped[-1] if folds_skipped else 0)
best_pt   = fold_dir(last_fold) / "best.pt"
val_tiles = fold_splits[last_fold]["val"]

if not best_pt.exists():
    print(f"  No best.pt for fold {last_fold} — skipping.")
else:
    model_vis, _ = build_faster_rcnn(
        NUM_CLASSES, ANCHOR_MIN_PX, ANCHOR_MAX_PX, ANCHOR_ASPECTS,
        trainable_backbone_layers=FROZEN_BACKBONE_LAYERS,
    )
    load_checkpoint(best_pt, model_vis)
    model_vis.to(DEVICE).eval()

    available = [t for t in val_tiles if (TILE_IMG_DIR / f"{t}.png").exists()]
    if not available:
        print("  No val tile images found.")
    else:
        sel = random.sample(available, min(6, len(available)))
        nr  = (len(sel) + 2) // 3
        fig, axes = plt.subplots(nr, 3, figsize=(20, 7 * nr))
        axes = np.array(axes).flatten()
        with torch.no_grad():
            for i, tn in enumerate(sel):
                ax = axes[i]
                ip = TILE_IMG_DIR / f"{tn}.png"
                img_bgr = cv2.imread(str(ip))
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                disp = img_rgb.copy()
                # Ground truth
                lp = TILE_LBL_FRCNN_DIR / f"{tn}.txt"
                ng = 0
                if lp.exists():
                    for ln in lp.read_text().strip().splitlines():
                        parts = ln.split()
                        if len(parts) == 5:
                            _, cx, cy, w, h = map(float, parts)
                            x1 = int((cx - w / 2) * TILE_SIZE)
                            y1 = int((cy - h / 2) * TILE_SIZE)
                            x2 = int((cx + w / 2) * TILE_SIZE)
                            y2 = int((cy + h / 2) * TILE_SIZE)
                            cv2.rectangle(disp, (x1, y1), (x2, y2), (0, 255, 0), 2)
                            ng += 1
                # Predictions
                img_t = TF.to_tensor(img_rgb).to(DEVICE)
                with torch.cuda.amp.autocast(enabled=(AMP and torch.cuda.is_available())):
                    out = model_vis([img_t])[0]
                n_pred = 0
                for b, s in zip(out["boxes"], out["scores"]):
                    if float(s) < CONF_THRESHOLD:
                        continue
                    x1, y1, x2, y2 = b.detach().cpu().numpy().astype(int)
                    cv2.rectangle(disp, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    n_pred += 1
                ax.imshow(disp)
                ax.set_title(f"GT:{ng}  Pred:{n_pred}")
                ax.axis("off")
        for j in range(i + 1, len(axes)):
            axes[j].axis("off")
        plt.suptitle(f"Fold {last_fold}: Green=GT, Red=Pred (Faster R-CNN)", fontsize=14)
        plt.tight_layout()
        plt.savefig(P["PLOTS"] / "predictions.png", dpi=150, bbox_inches="tight")
        plt.show()

    del model_vis
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("[10.1] Done.")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  10.2  Training curves
# ══════════════════════════════════════════════════════════════════
print("[10.2] Training curves ...")
hist_csv = fold_dir(last_fold) / "history.csv"
if not hist_csv.exists():
    print("  No history.csv")
else:
    df = pd.read_csv(hist_csv)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Total loss + RPN losses
    ax = axes[0, 0]
    if "loss" in df: ax.plot(df["epoch"], df["loss"], lw=2, label="total")
    ax.set_title("Train loss (total)"); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=.3)

    ax = axes[0, 1]
    for k in ["loss_objectness", "loss_rpn_box_reg"]:
        if k in df: ax.plot(df["epoch"], df[k], lw=2, label=k)
    ax.set_title("RPN losses"); ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(alpha=.3)

    ax = axes[0, 2]
    for k in ["loss_classifier", "loss_box_reg"]:
        if k in df: ax.plot(df["epoch"], df[k], lw=2, label=k)
    ax.set_title("ROI losses"); ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(alpha=.3)

    ax = axes[1, 0]
    for k in ["AP50", "AP50_95"]:
        if k in df: ax.plot(df["epoch"], df[k], lw=2, label=k)
    ax.set_title("mAP (validation)"); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=.3)

    ax = axes[1, 1]
    for k in ["Precision", "Recall"]:
        if k in df: ax.plot(df["epoch"], df[k], lw=2, label=k)
    ax.set_title("Precision & Recall @ IoU 0.5"); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=.3)

    ax = axes[1, 2]
    if "F1" in df: ax.plot(df["epoch"], df["F1"], lw=2, color="purple")
    ax.set_title("F1 @ IoU 0.5"); ax.set_xlabel("Epoch"); ax.grid(alpha=.3)

    plt.suptitle(f"Fold {last_fold} — Faster R-CNN Training Curves", fontsize=14)
    plt.tight_layout()
    plt.savefig(P["PLOTS"] / "curves.png", dpi=150, bbox_inches="tight")
    plt.show()
print("[10.2] Done.")


---
## 11. Error Analysis: Recall by Object Size and Scene Density


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  11.1  Per-size, per-density recall on a random val sample
# ══════════════════════════════════════════════════════════════════
print("[11.1] Error analysis ...")

if not best_pt.exists():
    print(f"  No best.pt for fold {last_fold} — skipping.")
else:
    model_err, _ = build_faster_rcnn(
        NUM_CLASSES, ANCHOR_MIN_PX, ANCHOR_MAX_PX, ANCHOR_ASPECTS,
        trainable_backbone_layers=FROZEN_BACKBONE_LAYERS,
    )
    load_checkpoint(best_pt, model_err)
    model_err.to(DEVICE).eval()

    vi = [TILE_IMG_DIR / f"{t}.png"
          for t in fold_splits[last_fold]["val"]
          if (TILE_IMG_DIR / f"{t}.png").exists()]
    vl = TILE_LBL_FRCNN_DIR

    ns = min(50 if DEBUG_MODE else 200, len(vi))
    samp = random.sample(vi, ns) if ns else []

    S = {
        "tp": 0, "fp": 0, "fn": 0, "gt": 0,
        "sz": defaultdict(lambda: {"gt": 0, "tp": 0}),
        "dn": defaultdict(lambda: {"gt": 0, "tp": 0}),
    }

    with torch.no_grad():
        for ip in tqdm(samp, desc="  Analysis"):
            lp = vl / (ip.stem + ".txt")
            gt = []
            if lp.exists():
                for ln in lp.read_text().strip().splitlines():
                    parts = ln.split()
                    if len(parts) == 5:
                        _, cx, cy, w, h = map(float, parts)
                        gt.append({"cx": cx, "cy": cy, "w": w, "h": h,
                                   "a": w * h * TILE_SIZE * TILE_SIZE})

            img_bgr = cv2.imread(str(ip))
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            img_t = TF.to_tensor(img_rgb).to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(AMP and torch.cuda.is_available())):
                out = model_err([img_t])[0]
            keep = out["scores"] >= CONF_THRESHOLD
            pb = out["boxes"][keep].detach().cpu().numpy()
            pds = []
            for x1, y1, x2, y2 in pb:
                pds.append({
                    "x1": x1 / TILE_SIZE, "y1": y1 / TILE_SIZE,
                    "x2": x2 / TILE_SIZE, "y2": y2 / TILE_SIZE,
                })

            S["gt"] += len(gt)
            mg, mp = set(), set()
            for pi, p in enumerate(pds):
                bi, bv = -1, 0
                for gi, g in enumerate(gt):
                    if gi in mg:
                        continue
                    gx1, gy1 = g["cx"] - g["w"] / 2, g["cy"] - g["h"] / 2
                    gx2, gy2 = g["cx"] + g["w"] / 2, g["cy"] + g["h"] / 2
                    ix = max(0, min(p["x2"], gx2) - max(p["x1"], gx1))
                    iy = max(0, min(p["y2"], gy2) - max(p["y1"], gy1))
                    inter = ix * iy
                    un = (p["x2"] - p["x1"]) * (p["y2"] - p["y1"]) + g["w"] * g["h"] - inter
                    iou = inter / max(un, 1e-9)
                    if iou > bv:
                        bv = iou; bi = gi
                if bv >= .5:
                    mg.add(bi); mp.add(pi); S["tp"] += 1
            S["fp"] += len(pds) - len(mp)
            S["fn"] += len(gt)  - len(mg)
            for gi, g in enumerate(gt):
                a = g["a"]
                bn = "<100" if a < 100 else "100-300" if a < 300 \
                     else "300-600" if a < 600 else "600+"
                S["sz"][bn]["gt"] += 1
                if gi in mg:
                    S["sz"][bn]["tp"] += 1
            d = "sparse" if len(gt) <= 5 else "moderate" if len(gt) <= 20 else "dense"
            S["dn"][d]["gt"] += len(gt)
            S["dn"][d]["tp"] += len(mg)

    pr = S["tp"] / max(S["tp"] + S["fp"], 1)
    rc = S["tp"] / max(S["tp"] + S["fn"], 1)
    f1 = 2 * pr * rc / max(pr + rc, 1e-9)
    print(f"  Precision={pr:.3f}  Recall={rc:.3f}  F1={f1:.3f}")
    print(f"  By size:")
    for b in ["<100", "100-300", "300-600", "600+"]:
        s = S["sz"][b]
        if s["gt"]:
            print(f"    {b:10s} {s['gt']:5d} GT  recall={s['tp'] / s['gt']:.3f}")
    print(f"  By density:")
    for d in ["sparse", "moderate", "dense"]:
        s = S["dn"][d]
        if s["gt"]:
            print(f"    {d:10s} {s['gt']:5d} GT  recall={s['tp'] / s['gt']:.3f}")
    with open(P["METRICS"] / "error_analysis.json", "w") as f:
        json.dump({
            "precision": pr, "recall": rc, "f1": f1,
            "by_size": {k: dict(v) for k, v in S["sz"].items()},
            "by_density": {k: dict(v) for k, v in S["dn"].items()},
        }, f, indent=2)

    del model_err
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("[11.1] Done.")


---
## 12. Run Summary


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  12.1  Save run summary
# ══════════════════════════════════════════════════════════════════
print("[12.1] Saving summary ...")

summary = {
    "paper":           "Yuan et al., An empirical analysis of deep learning methods for small object detection from satellite imagery",
    "model":           "Faster R-CNN (ResNet-50 + FPN)",
    "debug":           DEBUG_MODE,
    "folds":           NUM_FOLDS,
    "folds_completed": len(all_metrics) if "all_metrics" in dir() else 0,
    "epochs_max":      EPOCHS,
    "patience":        PATIENCE,
    "batch_micro":     BATCH_SIZE,
    "grad_accum":      GRAD_ACCUM,
    "batch_effective": BATCH_SIZE * GRAD_ACCUM,
    "lr":              LEARNING_RATE,
    "momentum":        MOMENTUM,
    "weight_decay":    WEIGHT_DECAY,
    "optimizer":       "SGD",
    "anchor_min_px":   ANCHOR_MIN_PX,
    "anchor_max_px":   ANCHOR_MAX_PX,
    "anchor_aspects":  ANCHOR_ASPECTS,
    "frozen_backbone_layers": FROZEN_BACKBONE_LAYERS,
    "tile_size":       TILE_SIZE,
    "img_size":        IMG_SIZE,
    "amp":             AMP,
    "workers":         NUM_WORKERS,
    "gpu":             HW["name"],
    "gpu_mem_gb":      HW["mem_gb"],
    "timestamp":       datetime.now().isoformat(),
}
with open(P["METRICS"] / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"  Saved: {P['METRICS'] / 'summary.json'}")
print(f"\n  {'=' * 50}")
print(f"  COMPLETE")
print(f"  {'=' * 50}")
print(f"  Mode  : {'DEBUG' if DEBUG_MODE else 'PAPER'}")
print(f"  Model : Faster R-CNN (ResNet-50 + FPN)")
print(f"  Folds : {len(all_metrics) if 'all_metrics' in dir() else '?'}/{NUM_FOLDS}")

if DEBUG_MODE:
    print(f"\n  Set DEBUG_MODE=False for paper results.")
    print(f"  Set CURRENT_FOLD='all' (or 0..{NUM_FOLDS - 1}) to train more folds.")

print("[12.1] Done.")
